# Augmentation ablations on NYUv2 (AUG-A … AUG-D)

This notebook explores data augmentations on NYUv2. Each run changes **one** augmentation against the 8-block baseline (8 trainable blocks, critic off, seed 42: `ckpt/seeds/nc8_s42` from `06_seed_repeats.ipynb`), and is judged with the decision rule of notebook 09.

| ID | Augmentation (change vs the baseline) | Source | Folder |
|---|---|---|---|
| — | **baseline:** random rescale s ∈ [1, 1.5] with depth ÷ s, crop 476×630, flip, photometric (brightness/contrast, gamma, HSV), patch mixing | SwinMTL (flip, photometric, patch mixing); the rescale is our choice | `ckpt/seeds/nc8_s42` |
| AUG-A | no rescale, crop **448×574** (SwinMTL's NYU geometry) | SwinMTL code | `ckpt/aug/aug_a` |
| AUG-B | rescale **s ∈ [0.5, 2.0]** (frames smaller than the crop are padded with invalid depth / ignored labels) | DPT §4.2 (segmentation training) | `ckpt/aug/aug_b` |
| AUG-C | random rotation **±2.5°** (uncovered corners invalid / ignored) | NYU depth practice (BTS, AdaBins) | `ckpt/aug/aug_c` |
| AUG-D | photometric random generator seeded per data-loader worker (`photo_rng_fix`): fixes a bug where all workers drew the same colour jitter every epoch | — (bug fix) | `ckpt/aug/aug_d` |

AUG-A was also run on 4 blocks (`ckpt/aug_4blk/aug_a`) as an interaction check.


## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_train import *
from repro_common import PAPER   # the papers' reference numbers, defined once (checked against the PDFs)

configure()
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'}")


Device: cuda, GPU: NVIDIA RTX A4000


## Training

In [3]:
AUG_RUNS = {
    "AUG-A": dict(overrides=dict(scale_aug=(1.0, 1.0)), crop=(448, 574)),
    "AUG-B": dict(overrides=dict(scale_aug=(0.5, 2.0)), crop=None),
    "AUG-C": dict(overrides=dict(rotate_deg=2.5), crop=None),
    "AUG-D": dict(overrides=dict(photo_rng_fix=True), crop=None),
}


def setup_aug(aug_id: str) -> None:
    spec = AUG_RUNS[aug_id]
    folder = aug_id.lower().replace("-", "_")
    configure(use_critic=False, seed=42,
              ckpt_root=ROOT / "ckpt" / "aug" / folder, results_root=ROOT / "results" / "aug" / folder, **spec["overrides"])
    CFG["variants"]["partial_ft8"] = dict(CFG["variants"]["partial_ft"], n_trainable_blocks=8)
    if spec["crop"]:
        CFG["datasets"]["nyu"]["crop"] = spec["crop"]


for aug_id in AUG_RUNS:
    setup_aug(aug_id)
    print(f"=== {aug_id}: scale_aug={CFG['scale_aug']}, crop={CFG['datasets']['nyu']['crop']}, "
          f"rotate_deg={CFG['rotate_deg']}, photo_rng_fix={CFG['photo_rng_fix']}")
    train_run("nyu", "partial_ft8")


=== AUG-A: scale_aug=(1.0, 1.0), crop=(448, 574), rotate_deg=0.0, photo_rng_fix=False
nyu_partial_ft8: already trained
=== AUG-B: scale_aug=(0.5, 2.0), crop=(476, 630), rotate_deg=0.0, photo_rng_fix=False
nyu_partial_ft8: already trained
=== AUG-C: scale_aug=(1.0, 1.5), crop=(476, 630), rotate_deg=2.5, photo_rng_fix=False
nyu_partial_ft8: already trained
=== AUG-D: scale_aug=(1.0, 1.5), crop=(476, 630), rotate_deg=0.0, photo_rng_fix=True
nyu_partial_ft8: already trained


## Evaluation
Full NYUv2 test set (654 images, Eigen crop) with notebook 03's protocols. Single pass is the result; sliding-window (TTA) is supplementary only. Every ablation is compared with the 8-block baseline and with the papers.
- Δ is the difference from the baseline. Δ / σ uses σ from notebook 06 (pooled over both arms, with the rule's floors: RMSE 0.003 m, mIoU 0.2), the same σ notebook 09 uses. |Δ| ≥ 2.5 σ counts as an effect.
- The mean log bias (prediction / GT) is also reported, since AUG-A is meant to probe it.
- Extra rows: E2 (4 blocks) and AUG-A on 4 blocks, from the stopped first attempt. They give AUG-A's effect at 4 blocks for comparison.
- For AUG-A the sliding-window (TTA) rows use 448×574 windows, its own training crop.
- Columns: SwinMTL's mIoU is as reported, mine is the standard metric (ΔmIoU uses it), with SwinMTL's per-image metric shown separately; DINOv2 trains its NYU depth head on ≈ 24k frames (frozen backbone), DPT-Hybrid is depth-pretrained on ≈ 1.4 M images (MIX 6) and fine-tuned on NYUv2; ours: 795 images.

In [4]:
def eval_aug(results_root: Path, ckpt_root: Path, crop=None, variant="partial_ft8", **overrides) -> Optional[dict]:
    configure(ckpt_root=ckpt_root, results_root=results_root, **overrides)
    CFG["variants"]["partial_ft8"] = dict(CFG["variants"]["partial_ft"], n_trainable_blocks=8)
    if crop:
        CFG["datasets"]["nyu"]["crop"] = crop
    out_path = CFG["results_root"] / "eval" / f"{run_name('nyu', variant)}.json"
    cached = load_json(out_path) if out_path.exists() else None
    if cached and "nyu_depth_slide" in cached and "log_bias" in cached:
        return cached
    if not (CFG["ckpt_root"] / run_name("nyu", variant) / "final.pt").exists():
        print(f"{ckpt_root}: not trained")
        return None
    model = load_trained("nyu", variant)
    results = cached if cached and "nyu_depth_slide" in cached else eval_nyu(model, slide=True)

    # mean log ratio prediction / GT (Eigen crop, 10 m), as in notebook 03 §8.9
    dcfg = CFG["datasets"]["nyu"]
    dataset = make_dataset("nyu", "test", train=False)
    biases = []
    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]
            gt = sample["depth"].to(device)
            gt = gt[0] if gt.ndim == 3 else gt
            H, W = gt.shape
            depth = predict(model, sample["image"][None].to(device))[0]
            pred = F.interpolate(depth, size=(H, W), mode="bilinear", align_corners=False)[0, 0].clamp(dcfg["min_depth"], dcfg["max_depth"])
            valid = (gt > dcfg["min_depth"]) & (gt < dcfg["max_depth"]) & eigen_crop_mask(H, W)
            biases.append((torch.log(pred[valid]) - torch.log(gt[valid])).mean().item())
    results["log_bias"] = float(np.mean(biases))
    save_json(results, out_path)
    del model
    torch.cuda.empty_cache()
    return results


BASE = "8 blocks (baseline)"
evaluated = {BASE: eval_aug(ROOT / "results" / "seeds" / "nc8_s42", ROOT / "ckpt" / "seeds" / "nc8_s42", use_critic=False, seed=42)}
for aug_id, spec in AUG_RUNS.items():
    folder = aug_id.lower().replace("-", "_")
    evaluated[aug_id] = eval_aug(ROOT / "results" / "aug" / folder, ROOT / "ckpt" / "aug" / folder, crop=spec["crop"],
                                 use_critic=False, seed=42, **spec["overrides"])
# extra rows from the stopped 4-block attempt (interaction with the block count)
extra = {"E2 (4 blocks)": eval_aug(ROOT / "results" / "nocritic", ROOT / "ckpt" / "nocritic", variant="partial_ft", use_critic=False),
         "AUG-A (4 blocks)": eval_aug(ROOT / "results" / "aug_4blk" / "aug_a", ROOT / "ckpt" / "aug_4blk" / "aug_a", crop=(448, 574),
                                      variant="partial_ft", use_critic=False, seed=42, scale_aug=(1.0, 1.0))}
configure()

# sigma as in notebook 09: pooled seed std of notebook 06, with the rule's floors
per_run_path = ROOT / "results" / "compare" / "seeds_nyu_per_run.csv"
SIGMA_FLOOR = {"RMSE": 0.003, "mIoU": 0.2}
sigma = {}
if per_run_path.exists():
    per_run = pd.read_csv(per_run_path)
    for col in ("RMSE", "mIoU"):
        groups = [g[col].to_numpy() for _, g in per_run.groupby("arm") if len(g) > 1]
        raw = float(np.sqrt(sum((len(a) - 1) * a.var(ddof=1) for a in groups) / sum(len(a) - 1 for a in groups)))
        sigma[col] = max(raw, SIGMA_FLOOR[col])
        print(f"sigma {col}: pooled {raw:.4g}, used {sigma[col]:.4g}")

rows = [
    {"model": "SwinMTL (Tab. III)", "RMSE": PAPER["swinmtl_tab3"]["SwinMTL"][0], "mIoU (paper: as reported; ours: standard)": PAPER["swinmtl_tab3"]["SwinMTL"][1]},
    {"model": "DINOv2 ViT-S/14 + DPT (Tab. 11; ≈ 24k training frames)", "RMSE": PAPER["dinov2_tab11_vits14"]["nyu"][2]},
    {"model": "DPT-Hybrid (Tab. 2; pretrained on ≈ 1.4 M images (MIX 6), fine-tuned on NYUv2)", "RMSE": PAPER["dpt_tab2"]["DPT-Hybrid"]["rmse"], "AbsRel": PAPER["dpt_tab2"]["DPT-Hybrid"]["abs_rel"], "δ1": PAPER["dpt_tab2"]["DPT-Hybrid"]["d1"]},
]
base = evaluated[BASE]
for label, R in list(evaluated.items()) + list(extra.items()):
    if not R:
        continue
    row = {"model": label, "RMSE": R["nyu_depth"]["rmse"], "AbsRel": R["nyu_depth"]["abs_rel"], "δ1": R["nyu_depth"]["d1"],
           "mIoU (paper: as reported; ours: standard)": R["nyu_seg"]["mIoU"], "mIoU (ours: SwinMTL per-image)": R["nyu_seg_swinmtl"]["mIoU"], "log bias": R.get("log_bias"),
           "RMSE +slide (TTA)": R["nyu_depth_slide"]["rmse"], "mIoU +slide (TTA)": R["nyu_seg_slide"]["mIoU"]}
    ref = extra["E2 (4 blocks)"] if label == "AUG-A (4 blocks)" else (None if label in (BASE, "E2 (4 blocks)") else base)
    if ref:
        row["ΔRMSE"] = row["RMSE"] - ref["nyu_depth"]["rmse"]
        row["ΔmIoU"] = row["mIoU (paper: as reported; ours: standard)"] - ref["nyu_seg"]["mIoU"]
        if sigma:
            row["ΔRMSE / σ"] = row["ΔRMSE"] / sigma["RMSE"]
            row["ΔmIoU / σ"] = row["ΔmIoU"] / sigma["mIoU"]
    rows.append(row)
aug_table = pd.DataFrame(rows).set_index("model")
aug_table.to_csv(ROOT / "results" / "compare" / "ablation_aug_nyu.csv")
display(aug_table.style.format(lambda v: "—" if pd.isna(v) else f"{v:.4f}" if abs(v) < 2 else f"{v:.2f}"))


sigma RMSE: pooled 0.001669, used 0.003
sigma mIoU: pooled 0.1389, used 0.2


,RMSE,mIoU (paper: as reported; ours: standard),AbsRel,δ1,mIoU (ours: SwinMTL per-image),log bias,RMSE +slide (TTA),mIoU +slide (TTA),ΔRMSE,ΔmIoU,ΔRMSE / σ,ΔmIoU / σ
model,,,,,,,,,,,,
SwinMTL (Tab. III),0.5179,58.14,—,—,—,—,—,—,—,—,—,—
DINOv2 ViT-S/14 + DPT (Tab. 11; ≈ 24k training frames),0.3560,—,—,—,—,—,—,—,—,—,—,—
"DPT-Hybrid (Tab. 2; pretrained on ≈ 1.4 M images (MIX 6), fine-tuned on NYUv2)",0.3570,—,0.1100,0.9040,—,—,—,—,—,—,—,—
8 blocks (baseline),0.4205,58.25,0.1215,0.8637,58.55,0.0208,0.4103,59.95,—,—,—,—
AUG-A,0.4365,56.45,0.1323,0.8495,57.91,0.0491,0.4167,58.48,0.0160,-1.7972,5.34,-8.99
AUG-B,0.4244,58.64,0.1224,0.8657,58.90,0.0267,0.4114,60.48,0.0039,0.3903,1.2934,1.9516
AUG-C,0.4183,58.17,0.1207,0.8635,58.52,0.0229,0.4076,59.82,-0.0022,-0.0762,-0.7300,-0.3808
AUG-D,0.4205,58.13,0.1201,0.8657,58.76,0.0153,0.4104,59.56,-0.0000,-0.1171,-0.0114,-0.5857
E2 (4 blocks),0.4282,57.39,0.1223,0.8601,58.23,0.0151,0.4174,59.03,—,—,—,—
